# Stage 5: statistics + explainability (XAI)
Uses the **fine-tuned models from Stage 4**. Nothing is trained here.

| Part | What it does | Needs | Time |
|---|---|---|---|
| **A. Stats** | Tables, 95% CIs, paired tests (McNemar + bootstrap), figures for all runs | CPU is fine | a few minutes |
| **B. Ablation** | Scrambles the marker letters and checks how much recovery drops | T4 GPU | ~10 min per model |
| **C. XAI** | Gradient x input, Integrated Gradients, GradientSHAP, attention: *where does the model look?* | T4 GPU | ~30-45 min per model |

1. Runtime > Change runtime type > **T4 GPU**.
2. In **Cell 1**, set `PARTS` and `ARMS` (split the models between team members). Change nothing else.
3. Runtime > **Run all**. Allow Google Drive access.
4. Paste back everything from each `STAGE 5 ... DONE` block (and the table printed above it).

Suggested split for Parts B + C:

| Person | ARMS |
|---|---|
| Member 1 | `marker_free,marker_endblock` |
| Member 2 | `marker_len06,marker_len06_n1` |
| Member 3 | `marker_len06_n8,marker_periodic` |

Part A only needs to be run **once** (by anyone): it reads every run in `results/stage4`.
Parts B + C save into your own folder `results/stage5/<your ARMS>/`, so all three can run at the same time.

### Cell 1: choose what to run (the ONLY thing you change)

In [ ]:
PARTS = ['stats', 'ablation', 'xai']        # any of 'stats', 'ablation', 'xai'
ARMS  = 'marker_free,marker_endblock'       # models for ablation + XAI (comma separated, no spaces)
                                            # e.g. marker_len06,marker_len06_n1   or   marker_len06_n8,marker_periodic

ALL_RUNS = ('marker_free', 'marker_periodic', 'marker_endblock',
            'marker_len04', 'marker_len05', 'marker_len06', 'marker_len07', 'marker_len12',
            'marker_len06_n1', 'marker_len06_n2', 'marker_len06_n8')
for a in ARMS.split(','):
    assert a in ALL_RUNS, f'{a} is misspelled'
assert set(PARTS) <= {'stats', 'ablation', 'xai'}, 'PARTS is misspelled'
print('Parts:', PARTS, '| models for B/C:', ARMS)

### Cell 2: mount Drive, set paths, check the Stage 4 models

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
PROJECT = '/content/drive/MyDrive/fyp_dna'
CODE    = f'{PROJECT}/code'
REPO    = '/content/Deep-DNA-based-storage'
OUT     = f'{PROJECT}/results/stage5'
OUT_BC  = f'{OUT}/' + ARMS.replace(',', '__')   # own folder per person, so parallel runs never overwrite each other
os.makedirs(CODE, exist_ok=True); os.makedirs(OUT, exist_ok=True)

runs = sorted(d for d in os.listdir(f'{PROJECT}/results/stage4')
              if os.path.isfile(f'{PROJECT}/results/stage4/{d}/predictions.csv'))
print('Stage 4 runs with test predictions:', ', '.join(runs))
missing = []
if 'ablation' in PARTS or 'xai' in PARTS:
    for a in ARMS.split(','):
        p = f'{PROJECT}/checkpoints/stage4/{a}/best.pt'
        print('found' if os.path.isfile(p) else 'MISSING', ' ', p)
        if not os.path.isfile(p):
            missing.append(p)
if missing:
    raise RuntimeError('STOP: fine-tuned model(s) missing from Drive:\n  ' + '\n  '.join(missing))
print('OK to continue.')
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'no GPU (fine for Part A only)'

### Cell 3: install packages (einops for DNAFormer, captum for Integrated Gradients / SHAP)

In [ ]:
!pip install -q einops captum

### Cell 4: write the Stage 1 helpers

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/stage1_generate.py
"""
Stage 1: paired dataset generator for the marker vs. marker-free DNAFormer study.

Makes three paired "arms" from the SAME random payloads and the SAME noise seeds:
  marker_free      128 payload bases, no markers
  marker_periodic  (24 payload + 8 marker) x 4   = 96 payload + 32 marker
  marker_endblock  96 payload, then the 4 markers together at the end (control)

Noisy reads come from DNAFormer's own simulator
(Deep-DNA-based-storage/DataGenerator/cluster_generator.py). We only give it our
own error rates; we do not edit the repository.

Usage (Colab or local):
  python stage1_generate.py --repo /content/Deep-DNA-based-storage --out /content/drive/MyDrive/fyp_dna/data/stage1_pilot
  python stage1_generate.py --repo ... --out ... --tiny      # quick smoke test
"""
import argparse
import csv
import hashlib
import json
import os
import random
import subprocess
import sys
import time

import numpy as np

# ---------------------------------------------------------------------------
# Fixed experiment settings (saved to config.json)
# ---------------------------------------------------------------------------
BASES = "ACGT"
STRAND_LEN = 128
MARKER = "ACGTCAGT"
MARKER_INTERVAL = 24          # payload bases between markers
N_MARKERS = 4
N_READS = 10
DELETION_LEVELS = [0.01, 0.03, 0.05, 0.08]
INSERTION_RATE = 0.01
SUBSTITUTION_RATE = 0.01
LONG_DELETION_RATE = 0.0      # all deletions are single-base; change later if wanted

ARMS = ["marker_free", "marker_periodic", "marker_endblock"]   # Stage 1 default

# Every arm: how the 128-base strand is built. "periodic" = n blocks of 128/n bases
# (n = 4 unless the arm sets "n_markers"), each block = (128/n - L) data bases followed by
# the L-base marker. Payloads are always
# the FIRST data bases of the same 128-base random payload, and the noise seed is
# shared, so all arms are paired (same data, same error positions).
BLOCK = STRAND_LEN // N_MARKERS   # 32 (default block; use arm_block(arm) per arm)
ARM_SPECS = {
    "marker_free":     {"layout": "none",      "marker": ""},
    "marker_periodic": {"layout": "periodic",  "marker": "ACGTCAGT"},      # length 8 (Stage 4)
    "marker_endblock": {"layout": "end_block", "marker": "ACGTCAGT"},      # control
    "marker_len04":    {"layout": "periodic",  "marker": "CAGT"},
    "marker_len05":    {"layout": "periodic",  "marker": "TCATG"},
    "marker_len06":    {"layout": "periodic",  "marker": "CTGTCA"},
    "marker_len07":    {"layout": "periodic",  "marker": "CTGATCA"},
    "marker_len12":    {"layout": "periodic",  "marker": "ACGTCAGTGCAT"},
    # Spacing study: same 6-letter marker, 2 or 8 markers per strand (4 = marker_len06)
    "marker_len06_n2": {"layout": "periodic",  "marker": "CTGTCA", "n_markers": 2},   # 2 x (58 + 6)
    "marker_len06_n8": {"layout": "periodic",  "marker": "CTGTCA", "n_markers": 8},   # 8 x (10 + 6)
    "marker_len06_n1": {"layout": "middle",    "marker": "CTGTCA", "n_markers": 1},   # 61 + 6 + 61
}
LENGTH_ARMS = ["marker_len04", "marker_len05", "marker_len06", "marker_len07", "marker_len12"]
SPACING_ARMS = ["marker_len06_n2", "marker_len06_n8"]
MIDDLE_ARMS = ["marker_len06_n1"]   # one marker in the middle of the strand
ALL_ARMS = list(ARM_SPECS)

# Seed ranges never overlap between splits, so the same payload cannot appear in two splits.
SPLIT_SEED_BASE = {"train": 10_000_000, "val": 20_000_000, "test": 30_000_000}
LEVEL_SEED_STRIDE = 1_000_000  # each deletion level gets its own block of seeds
NOISE_SEED_OFFSET = 1_000_000_000
SHUFFLE_SEED = 12345

PILOT_SIZES = {"train": 2500, "val": 250, "test": 1000}   # clusters PER deletion level
TINY_SIZES = {"train": 20, "val": 5, "test": 10}


def level_tag(d):
    return "del%02d" % round(d * 100)


# ---------------------------------------------------------------------------
# Strand construction
# ---------------------------------------------------------------------------
def make_payload128(payload_seed):
    rng = np.random.default_rng(payload_seed)
    return "".join(BASES[i] for i in rng.integers(0, 4, STRAND_LEN))


def arm_marker(arm):
    return ARM_SPECS[arm]["marker"]


def arm_n_markers(arm):
    return ARM_SPECS[arm].get("n_markers", N_MARKERS) if ARM_SPECS[arm]["layout"] != "none" else 0


def arm_block(arm):
    n = arm_n_markers(arm) or N_MARKERS
    assert STRAND_LEN % n == 0
    return STRAND_LEN // n


def arm_n_data(arm):
    return STRAND_LEN - arm_n_markers(arm) * len(arm_marker(arm))


def data_per_block(arm):
    return arm_block(arm) - len(arm_marker(arm))


def build_strand(arm, payload128):
    """Return (clean_strand, payload, marker_positions)."""
    spec = ARM_SPECS[arm]
    m = spec["marker"]
    if spec["layout"] == "none":
        return payload128, payload128, []
    n_data, nm = arm_n_data(arm), arm_n_markers(arm)
    payload = payload128[:n_data]               # same seed -> first bases shared by all arms
    if spec["layout"] == "periodic":
        d = data_per_block(arm)
        parts, positions = [], []
        for k in range(nm):
            parts.append(payload[k * d:(k + 1) * d])
            start = sum(len(p) for p in parts)
            parts.append(m)
            positions.append([start, start + len(m)])
        return "".join(parts), payload, positions
    if spec["layout"] == "middle":
        h = n_data // 2
        return payload[:h] + m + payload[h:], payload, [[h, h + len(m)]]
    if spec["layout"] == "end_block":
        start = len(payload)
        positions = [[start + k * len(m), start + (k + 1) * len(m)] for k in range(nm)]
        return payload + m * nm, payload, positions
    raise ValueError(arm)


def remove_markers(arm, strand):
    """Payload-only view of a 128-base strand (used by later stages for scoring)."""
    spec = ARM_SPECS[arm]
    if spec["layout"] == "none":
        return strand
    n_data = arm_n_data(arm)
    if spec["layout"] == "periodic":
        d, b = data_per_block(arm), arm_block(arm)
        return "".join(strand[k * b:k * b + d] for k in range(arm_n_markers(arm)))
    if spec["layout"] == "middle":
        h = n_data // 2
        return strand[:h] + strand[h + len(spec["marker"]):]
    if spec["layout"] == "end_block":
        return strand[:n_data]
    raise ValueError(arm)


def payload_to_bytes(payload):
    """2 bits per base (A=00 C=01 G=10 T=11). 128 bases -> 32 bytes, 96 bases -> 24 bytes."""
    bits = "".join(format(BASES.index(b), "02b") for b in payload)
    return bytes(int(bits[i:i + 8], 2) for i in range(0, len(bits), 8))


# ---------------------------------------------------------------------------
# DNAFormer simulator wrapper
# ---------------------------------------------------------------------------
def load_simulator(repo):
    gen_dir = os.path.join(repo, "DataGenerator")
    if not os.path.isfile(os.path.join(gen_dir, "cluster_generator.py")):
        sys.exit("ERROR: cluster_generator.py not found in %s. Check --repo." % gen_dir)
    sys.path.insert(0, gen_dir)
    from cluster_generator import ClusterGenerator  # noqa: E402
    return ClusterGenerator


def error_rates(deletion_rate):
    """Rates in the simulator's own format.

    The simulator gives each base one error chance equal to the SUM of the general
    rates, then picks the error type with the per-base weights. Using the same numbers
    for both makes the per-base probabilities exactly sub/ins/del.
    """
    general = {"s": SUBSTITUTION_RATE, "i": INSERTION_RATE, "d": deletion_rate, "ld": LONG_DELETION_RATE}
    per_base = {b: {"s": SUBSTITUTION_RATE, "i": 0.25, "pi": INSERTION_RATE,
                    "d": deletion_rate, "ld": LONG_DELETION_RATE} for b in BASES}
    # 'i' only chooses WHICH base is inserted (uniform here); 'pi' is the insertion rate.
    return general, per_base


def simulate_reads(ClusterGenerator, clean, deletion_rate, noise_seed):
    random.seed(noise_seed)          # the simulator uses Python's global random module
    np.random.seed(noise_seed % (2 ** 32))
    general, per_base = error_rates(deletion_rate)
    gen = ClusterGenerator(general, per_base, clean, min_copies=N_READS, max_copies=N_READS)
    gen.cluster_size = N_READS       # the repo's randint(min, max+1) could give 11; force exactly 10
    gen.generate_cluster(delta=0)    # delta=0: no random jitter of the rates
    assert len(gen.copies) == N_READS
    return list(gen.copies)


# ---------------------------------------------------------------------------
# Cluster plan
# ---------------------------------------------------------------------------
def plan_clusters(sizes):
    """List of dicts: split, subset, deletion_rate, payload_seed, noise_seed, pair_id."""
    plan = []
    for split in ["train", "val", "test"]:
        for li, d in enumerate(DELETION_LEVELS):
            for i in range(sizes[split]):
                seed = SPLIT_SEED_BASE[split] + li * LEVEL_SEED_STRIDE + i
                subset = split if split != "test" else "test_" + level_tag(d)
                plan.append({
                    "pair_id": "%s_%s_%06d" % (split, level_tag(d), i),
                    "split": split,
                    "subset": subset,
                    "deletion_rate": d,
                    "payload_seed": seed,
                    "noise_seed": seed + NOISE_SEED_OFFSET,
                })
    # Mix the deletion levels inside train and val (fixed shuffle seed).
    rng = random.Random(SHUFFLE_SEED)
    for split in ["train", "val"]:
        idx = [k for k, c in enumerate(plan) if c["split"] == split]
        items = [plan[k] for k in idx]
        rng.shuffle(items)
        for k, c in zip(idx, items):
            plan[k] = c
    return plan


def make_cluster(ClusterGenerator, arm, c):
    payload128 = make_payload128(c["payload_seed"])
    clean, payload, marker_pos = build_strand(arm, payload128)
    reads = simulate_reads(ClusterGenerator, clean, c["deletion_rate"], c["noise_seed"])
    n_marker = len(clean) - len(payload)
    return {
        "cluster_id": "%s__%s" % (arm, c["pair_id"]),
        "pair_id": c["pair_id"],
        "arm": arm,
        "split": c["split"],
        "subset": c["subset"],
        "seed": c["payload_seed"],
        "payload_seed": c["payload_seed"],
        "noise_seed": c["noise_seed"],
        "marker_present": ARM_SPECS[arm]["layout"] != "none",
        "marker_layout": ARM_SPECS[arm]["layout"],
        "marker_sequence": arm_marker(arm),
        "marker_interval": data_per_block(arm) if ARM_SPECS[arm]["layout"] == "periodic" else None,
        "marker_positions": marker_pos,
        "marker_overhead": n_marker / len(clean),
        "payload_length": len(payload),
        "strand_length": len(clean),
        "clean_strand": clean,
        "payload_strand": payload,
        "payload_bytes_hex": payload_to_bytes(payload).hex(),
        "noisy_reads": reads,
        "read_lengths": [len(r) for r in reads],
        "deletion_rate": c["deletion_rate"],
        "insertion_rate": INSERTION_RATE,
        "substitution_rate": SUBSTITUTION_RATE,
        "long_deletion_rate": LONG_DELETION_RATE,
    }


# ---------------------------------------------------------------------------
# Writing
# ---------------------------------------------------------------------------
def write_subset(folder, clusters):
    os.makedirs(folder, exist_ok=True)
    with open(os.path.join(folder, "clean.fasta"), "w", newline="\n") as f:
        for c in clusters:
            f.write(">%s\n%s\n" % (c["cluster_id"], c["clean_strand"]))
    with open(os.path.join(folder, "noisy_reads.fasta"), "w", newline="\n") as f:
        for c in clusters:
            for j, r in enumerate(c["noisy_reads"]):
                f.write(">%s|read_%d\n%s\n" % (c["cluster_id"], j, r))
    with open(os.path.join(folder, "metadata.json"), "w", newline="\n") as f:
        json.dump(clusters, f, separators=(",", ":"))


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def repo_commit(repo):
    try:
        return subprocess.check_output(["git", "-C", repo, "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True, help="path to Deep-DNA-based-storage")
    ap.add_argument("--out", required=True, help="output folder")
    ap.add_argument("--tiny", action="store_true", help="very small smoke-test dataset")
    ap.add_argument("--arms", default=",".join(ARMS),
                    help="comma-separated arms to build (default: the 3 Stage 1 arms); "
                         "'lengths' = the marker-length arms %s; 'spacing' = %s; 'middle' = %s" % (LENGTH_ARMS, SPACING_ARMS, MIDDLE_ARMS))
    args = ap.parse_args()
    groups = {"lengths": LENGTH_ARMS, "spacing": SPACING_ARMS, "middle": MIDDLE_ARMS}
    arms = groups.get(args.arms) or [a.strip() for a in args.arms.split(",")]
    for a in arms:
        if a not in ARM_SPECS:
            sys.exit("ERROR: unknown arm %s. Known: %s" % (a, ALL_ARMS))

    sizes = TINY_SIZES if args.tiny else PILOT_SIZES
    ClusterGenerator = load_simulator(args.repo)
    plan = plan_clusters(sizes)
    os.makedirs(args.out, exist_ok=True)

    t0 = time.time()
    manifest_rows = []
    for arm in arms:
        by_subset = {}
        for k, c in enumerate(plan):
            cl = make_cluster(ClusterGenerator, arm, c)
            by_subset.setdefault(c["subset"], []).append(cl)
            if (k + 1) % 2000 == 0:
                print("  %s: %d/%d clusters (%.0fs)" % (arm, k + 1, len(plan), time.time() - t0), flush=True)
        for subset, clusters in by_subset.items():
            folder = os.path.join(args.out, arm, subset)
            write_subset(folder, clusters)
            for cl in clusters:
                manifest_rows.append({
                    "cluster_id": cl["cluster_id"], "pair_id": cl["pair_id"], "arm": arm,
                    "split": cl["split"], "subset": subset,
                    "deletion_rate": cl["deletion_rate"], "insertion_rate": cl["insertion_rate"],
                    "substitution_rate": cl["substitution_rate"],
                    "payload_seed": cl["payload_seed"], "noise_seed": cl["noise_seed"],
                    "marker_present": cl["marker_present"], "marker_layout": cl["marker_layout"],
                    "marker_sequence": cl["marker_sequence"], "marker_interval": cl["marker_interval"],
                    "payload_length": cl["payload_length"], "strand_length": cl["strand_length"],
                    "marker_overhead": cl["marker_overhead"], "n_reads": len(cl["noisy_reads"]),
                    "mean_read_length": round(float(np.mean(cl["read_lengths"])), 2),
                    "folder": os.path.join(arm, subset).replace("\\", "/"),
                })
        print("%s done (%.0fs)" % (arm, time.time() - t0), flush=True)

    with open(os.path.join(args.out, "dataset_manifest.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(manifest_rows[0].keys()))
        w.writeheader()
        w.writerows(manifest_rows)

    config = {
        "stage": 1, "mode": "tiny" if args.tiny else "pilot",
        "clusters_per_level": sizes, "deletion_levels": DELETION_LEVELS,
        "insertion_rate": INSERTION_RATE, "substitution_rate": SUBSTITUTION_RATE,
        "long_deletion_rate": LONG_DELETION_RATE, "strand_length": STRAND_LEN,
        "marker": MARKER, "marker_interval": MARKER_INTERVAL, "n_markers": N_MARKERS,
        "n_reads": N_READS, "arms": arms,
        "arm_specs": {a: dict(ARM_SPECS[a], data_bases=len(build_strand(a, "A" * STRAND_LEN)[1]))
                      for a in arms},
        "split_seed_base": SPLIT_SEED_BASE,
        "level_seed_stride": LEVEL_SEED_STRIDE, "noise_seed_offset": NOISE_SEED_OFFSET,
        "shuffle_seed": SHUFFLE_SEED, "simulator": "DataGenerator/cluster_generator.py",
        "simulator_repo_commit": repo_commit(args.repo),
        "python": sys.version.split()[0], "numpy": np.__version__,
    }
    with open(os.path.join(args.out, "config.json"), "w", newline="\n") as f:
        json.dump(config, f, indent=2)

    checksums = {}
    for root, _, files in os.walk(args.out):
        for name in files:
            if name in ("checksums.json",):
                continue
            p = os.path.join(root, name)
            checksums[os.path.relpath(p, args.out).replace("\\", "/")] = sha256(p)
    with open(os.path.join(args.out, "checksums.json"), "w", newline="\n") as f:
        json.dump(dict(sorted(checksums.items())), f, indent=1)

    print("\nDONE: %d clusters per arm x %d arms in %.0fs -> %s"
          % (len(plan), len(arms), time.time() - t0, args.out))


if __name__ == "__main__":
    main()

### Cell 5: write the DNAFormer adapter

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/dnaf_adapter.py
"""
Minimal adapter between our Stage 1 data and the UNCHANGED DNAFormer model.

What it does:
  - builds DNAFormer exactly as the repo defines it (DNN/model_DNAFormer_siamese.py)
  - loads the pretrained checkpoint and REFUSES to continue if any weight is missing
    (the repo uses strict=False, which silently leaves layers random)
  - turns a cluster of reads into the model's input tensors, without the repo
    loader's problems (crash on Python >= 3.11, reads of length 132 turned into
    zeros, deletion-heavy reads thrown away, clusters silently swapped)

The model architecture is not touched. Input/target lengths stay 132 / 128.
"""
import os
import sys

import numpy as np

BASES = "ACGT"
BASE_IDX = {b: i for i, b in enumerate(BASES)}   # same mapping as the repo: A=0 C=1 G=2 T=3

LABEL_LEN = 128        # repo: label_length 140 - index_length 12
INPUT_LEN = 132        # repo: noisy_copies_length = 140 + 4 - 12
MAX_READS = 16         # repo: max_number_per_cluster
ORIG_MAX_DEVIATION = 4  # repo: corrupt_max_deviation

# Old checkpoint name -> current model name (found in Stage 2: same shape, renamed in code)
CHECKPOINT_RENAMES = {
    "fusion.pred_fusion": "fusion.pred_fusion_left",
    "fusion.pred_fusion_flip": "fusion.pred_fusion_right",
}


def reset_fusion_weights(model):
    """Set both fusion vectors to 1.0, i.e. what the repo's own loader ends up using."""
    import torch
    with torch.no_grad():
        model.fusion.pred_fusion_left.fill_(1.0)
        model.fusion.pred_fusion_right.fill_(1.0)


class DNAFormerConfig:
    """Same values as DNN/inference.py in the repo."""
    index_length = 12
    label_length = 140
    filter_index = True
    corrupt_max_deviation = 4
    noisy_copies_length = INPUT_LEN
    max_number_per_cluster = MAX_READS
    read_padding = "end"
    model_config = "siamese"
    n_head = 32
    activation = "gelu"
    num_layers = 12
    d_model = 1024
    alignment_filters = 128
    alignement_filters = 128   # the repo's train.py uses this misspelling
    dim_feedforward = 2048
    output_ch = 4
    enc_filters = 4
    p_dropout = 0
    class_token = 0
    use_input_scaling = False
    device = "cpu"


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
def build_model(repo, device="cpu"):
    sys.path.insert(0, os.path.join(repo, "DNN"))
    from model_DNAFormer_siamese import net  # noqa: E402
    cfg = DNAFormerConfig()
    cfg.device = device
    return net(cfg).to(device), cfg


def load_checkpoint(model, path, device="cpu"):
    """Load weights strictly. Returns a report dict; raises if anything does not match."""
    import torch
    try:
        ckpt = torch.load(path, map_location=device, weights_only=True)
        loader = "weights_only=True"
    except Exception:
        # Official checkpoint from the DNAFormer authors; may contain non-tensor objects.
        ckpt = torch.load(path, map_location=device, weights_only=False)
        loader = "weights_only=False"
    state = ckpt.get("model_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    state = {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}
    # The released checkpoint uses older names for the two fusion weight vectors.
    # The repo's strict=False loading silently skips them (they stay at 1.0).
    # We map them explicitly; the shape check below still guards the mapping.
    renamed = {}
    for old, new in CHECKPOINT_RENAMES.items():
        if old in state and new not in state:
            state[new] = state.pop(old)
            renamed[old] = new
    model_keys = set(model.state_dict().keys())
    ckpt_keys = set(state.keys())
    missing = sorted(model_keys - ckpt_keys)
    unexpected = sorted(ckpt_keys - model_keys)
    shape_bad = sorted(k for k in model_keys & ckpt_keys
                       if tuple(model.state_dict()[k].shape) != tuple(state[k].shape))
    report = {
        "loader": loader,
        "checkpoint_top_keys": sorted(ckpt.keys()) if isinstance(ckpt, dict) else [],
        "checkpoint_epoch": ckpt.get("epoch") if isinstance(ckpt, dict) else None,
        "n_tensors": len(ckpt_keys), "missing": missing, "unexpected": unexpected,
        "shape_mismatch": shape_bad, "renamed": renamed,
        "renamed_shapes": {new: tuple(state[new].shape) for new in renamed.values()},
        "renamed_stats": {new: [round(float(state[new].min()), 4), round(float(state[new].mean()), 4),
                                round(float(state[new].max()), 4)] for new in renamed.values()},
    }
    if missing or unexpected or shape_bad:
        raise RuntimeError("Checkpoint does not match the model: %s" % report)
    model.load_state_dict(state, strict=True)
    return report


# ---------------------------------------------------------------------------
# Input encoding
# ---------------------------------------------------------------------------
def select_reads(reads, policy="relaxed"):
    """Which reads go into the model.

    relaxed  (ours): keep every read; reads longer than 132 are cut to 132.
    original (repo): keep reads with |len-128| <= 4; a read of exactly 132 becomes
                     an all-zero row (repo bug). If no read survives, the repo would
                     replace the whole cluster with a random other one.
    """
    if policy == "relaxed":
        return [r[:INPUT_LEN] for r in reads[:MAX_READS]]
    if policy == "original":
        kept = [r for r in reads if abs(len(r) - LABEL_LEN) <= ORIG_MAX_DEVIATION]
        return [r if len(r) < INPUT_LEN else "" for r in kept[:MAX_READS]]
    raise ValueError(policy)


def one_hot(read, length=INPUT_LEN):
    x = np.zeros((4, length), dtype=np.float32)
    for j, b in enumerate(read[:length]):
        x[BASE_IDX[b], j] = 1.0
    return x


def encode_cluster(reads, policy="relaxed"):
    """Return (left, right, n_used) as numpy arrays of shape [16, 4, 132].

    left  = each read one-hot, zero-padded at the end (repo read_padding='end')
    right = each read reversed, then zero-padded at the end (repo siamese branch)
    Unused read slots stay all-zero, as in the repo.
    """
    used = select_reads(reads, policy)
    left = np.zeros((MAX_READS, 4, INPUT_LEN), dtype=np.float32)
    right = np.zeros((MAX_READS, 4, INPUT_LEN), dtype=np.float32)
    for i, r in enumerate(used):
        left[i] = one_hot(r)
        right[i] = one_hot(r[::-1])
    return left, right, len(used)


def make_batch(clusters, policy="relaxed", device="cpu"):
    """clusters: list of Stage 1 metadata dicts. Returns (model_input, labels).

    model_input has shape [2B, 16, 4, 132]: left branch first, right branch second,
    exactly as the repo concatenates them. labels has shape [B, 128] (class indices).
    """
    import torch
    L, R, Y = [], [], []
    for c in clusters:
        left, right, _ = encode_cluster(c["noisy_reads"], policy)
        L.append(left)
        R.append(right)
        Y.append([BASE_IDX[b] for b in c["clean_strand"]])
    x = torch.from_numpy(np.concatenate([np.stack(L), np.stack(R)], axis=0)).to(device)
    y = torch.tensor(Y, dtype=torch.long, device=device)
    return x, y


def decode(pred_logits):
    """[B, 4, 128] logits -> list of DNA strings."""
    idx = pred_logits.argmax(dim=1).cpu().numpy()
    return ["".join(BASES[i] for i in row) for row in idx]


# ---------------------------------------------------------------------------
# Scoring helpers
# ---------------------------------------------------------------------------
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

### Cell 6: write the Stage 3 scoring code

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/stage3_baselines.py
"""
Stage 3: baseline evaluation on all test clusters. No training.

Methods
  mv            plain majority vote: line the reads up from the left, vote per position.
                Run on all 3 arms.
  mv_sync       marker-sync majority vote (periodic arm only): find the 4 markers in each
                read, cut the read into the 4 payload segments between them, then vote
                inside each segment. Each marker lets the vote re-align after deletions.
  dnaformer     pretrained DNAFormer, no fine-tuning, relaxed read policy (all reads kept).
                Run on all 3 arms.

Scoring is payload-only: marker bases are removed from the prediction before comparing it
with the true payload (128 bases for marker_free, 96 for the marker arms).

Metrics per (method, arm, deletion level)
  exact recovery rate with 95% Wilson CI
  normalized edit distance (edit distance / payload length), mean with 95% CI
  per-base accuracy (same base at the same payload position), mean with 95% CI
  file-level recovery: 25 consecutive test strands = 1 "file"; a file counts as
    recovered only if all 25 strands are exact (no outer error-correcting code), Wilson CI
  runtime (s total, ms per cluster) and peak GPU memory (DNAFormer only)

Outputs (in --out)
  predictions.csv   one row per (method, arm, cluster)
  results.csv       one row per (method, arm, deletion level)
  stage3_log.json   environment, checkpoint report, settings

Usage
  python stage3_baselines.py --repo REPO --data DATA --ckpt CKPT --out OUT
  python stage3_baselines.py --repo REPO --data DATA --out OUT --no-torch    # majority votes only
"""
import argparse
import csv
import json
import math
import os
import platform
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import dnaf_adapter as A  # noqa: E402
import stage1_generate as G  # noqa: E402

FILE_SIZE = 25          # strands per "file" for file-level recovery
MARKER_MAX_ED = 2       # a marker match may have at most this many edits
MARKER_SEARCH = 6       # search +/- this many bases around where the marker should be
DNAF_BATCH = 32
Z95 = 1.959964


# ---------------------------------------------------------------------------
# Majority vote
# ---------------------------------------------------------------------------
def vote(columns_reads, length):
    """Left-aligned positional majority vote over a list of strings.

    Position j is voted on by every string that is long enough. Ties go to the first base
    in A, C, G, T order. Positions no string reaches get 'A' (counted as a guess)."""
    out = []
    for j in range(length):
        counts = [0, 0, 0, 0]
        for r in columns_reads:
            if j < len(r):
                counts[A.BASE_IDX[r[j]]] += 1
        out.append(A.BASES[int(np.argmax(counts))])
    return "".join(out)


def mv_plain(reads):
    return vote(reads, G.STRAND_LEN)


def find_marker(read, expected_start):
    """Best approximate match of the marker near expected_start.

    Semi-global alignment: the marker must match completely, the read may start and end
    anywhere inside the search window. Returns (start, end, edits) or None if the best
    match has more than MARKER_MAX_ED edits."""
    m = G.MARKER
    lo = max(0, expected_start - MARKER_SEARCH)
    hi = min(len(read), expected_start + len(m) + MARKER_SEARCH)
    text = read[lo:hi]
    if not text:
        return None
    exact = text.find(m)
    if exact >= 0:
        # take the exact hit closest to the expected position
        best, s = None, exact
        while s >= 0:
            if best is None or abs(lo + s - expected_start) < abs(lo + best - expected_start):
                best = s
            s = text.find(m, s + 1)
        return lo + best, lo + best + len(m), 0
    # D[i][j]: edits to align marker[:i] with a substring of text ending at j
    n = len(text)
    prev = [0] * (n + 1)              # row i=0: free start anywhere
    prev_start = list(range(n + 1))   # where that alignment starts in text
    for i in range(1, len(m) + 1):
        cur = [i] + [0] * n
        cur_start = [0] + [0] * n
        for j in range(1, n + 1):
            sub = prev[j - 1] + (m[i - 1] != text[j - 1])
            dele = prev[j] + 1        # marker base missing from read
            ins = cur[j - 1] + 1      # extra base in read
            best = min(sub, dele, ins)
            cur[j] = best
            if best == sub:
                cur_start[j] = prev_start[j - 1]
            elif best == dele:
                cur_start[j] = prev_start[j]
            else:
                cur_start[j] = cur_start[j - 1]
        prev, prev_start = cur, cur_start
    best_j = min(range(1, n + 1), key=lambda j: (prev[j], abs(lo + j - len(m) - expected_start)))
    if prev[best_j] > MARKER_MAX_ED:
        return None
    return lo + prev_start[best_j], lo + best_j, prev[best_j]


def split_at_markers(read):
    """Cut one periodic-arm read into its 4 payload segments.

    Markers are searched one after the other; the expected position of the next marker is
    24 bases after the end of the previous one, so a deletion earlier in the read does not
    throw off the later searches. If a marker is not found, we assume it sits where expected.
    Returns (segments, n_markers_found)."""
    segments, found, pos = [], 0, 0
    for _ in range(G.N_MARKERS):
        expected = pos + G.MARKER_INTERVAL
        hit = find_marker(read, expected)
        if hit is None:
            start, end = min(expected, len(read)), min(expected + len(G.MARKER), len(read))
        else:
            start, end, _ = hit
            found += 1
        segments.append(read[pos:start])
        pos = end
    return segments, found


def mv_sync(reads):
    """Marker-sync majority vote. Returns (payload96, fraction of markers found)."""
    per_read = [split_at_markers(r) for r in reads]
    payload = "".join(vote([segs[k] for segs, _ in per_read], G.MARKER_INTERVAL)
                      for k in range(G.N_MARKERS))
    found = sum(f for _, f in per_read) / (G.N_MARKERS * len(reads))
    return payload, found


# ---------------------------------------------------------------------------
# Scoring and statistics
# ---------------------------------------------------------------------------
def score(pred_payload, true_payload):
    ed = A.edit_distance(pred_payload, true_payload)
    same = sum(a == b for a, b in zip(pred_payload, true_payload))
    n = len(true_payload)
    return {"edit_distance": ed, "ned": ed / n, "base_acc": same / n, "exact": int(pred_payload == true_payload)}


def wilson(k, n):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    den = 1 + Z95 ** 2 / n
    mid = (p + Z95 ** 2 / (2 * n)) / den
    half = Z95 * math.sqrt(p * (1 - p) / n + Z95 ** 2 / (4 * n * n)) / den
    return max(0.0, mid - half), min(1.0, mid + half)


def mean_ci(x):
    x = np.asarray(x, dtype=float)
    m = float(x.mean())
    half = Z95 * float(x.std(ddof=1)) / math.sqrt(len(x)) if len(x) > 1 else float("nan")
    return m, m - half, m + half


def summarize(method, arm, level, rows, runtime_s, gpu_peak_gb):
    exact = [r["exact"] for r in rows]
    k, n = sum(exact), len(exact)
    lo, hi = wilson(k, n)
    ned_m, ned_lo, ned_hi = mean_ci([r["ned"] for r in rows])
    acc_m, acc_lo, acc_hi = mean_ci([r["base_acc"] for r in rows])
    files = {}
    for r in rows:
        files.setdefault(r["file_id"], []).append(r["exact"])
    full = [f for f in files.values() if len(f) == FILE_SIZE]
    fk = sum(all(f) for f in full)
    flo, fhi = wilson(fk, len(full))
    payload_len = rows[0]["payload_length"]
    return {
        "method": method, "arm": arm, "deletion_rate": level, "n_clusters": n,
        "payload_length": payload_len, "info_rate": payload_len / G.STRAND_LEN,
        "exact_recovery": k / n, "exact_ci_low": lo, "exact_ci_high": hi,
        "ned_mean": ned_m, "ned_ci_low": ned_lo, "ned_ci_high": ned_hi,
        "base_acc_mean": acc_m, "base_acc_ci_low": acc_lo, "base_acc_ci_high": acc_hi,
        "n_files": len(full), "files_recovered": fk,
        "file_recovery": fk / len(full) if full else float("nan"),
        "file_ci_low": flo, "file_ci_high": fhi,
        "runtime_s": runtime_s, "ms_per_cluster": 1000 * runtime_s / n,
        "gpu_peak_gb": gpu_peak_gb,
        "markers_found": (float(np.mean([r["markers_found"] for r in rows]))
                          if rows[0]["markers_found"] != "" else ""),
    }


# ---------------------------------------------------------------------------
# Runner
# ---------------------------------------------------------------------------
PRED_FIELDS = ["method", "arm", "deletion_rate", "cluster_id", "pair_id", "file_id", "payload_length",
               "true_payload", "pred_payload", "edit_distance", "ned", "base_acc", "exact",
               "markers_found"]


def load_test(data, arm, level, limit):
    with open(os.path.join(data, arm, "test_" + G.level_tag(level), "metadata.json")) as f:
        clusters = json.load(f)
    return clusters[:limit] if limit else clusters


def make_rows(method, arm, level, clusters, preds, markers_found=None):
    rows = []
    for i, (c, p) in enumerate(zip(clusters, preds)):
        s = score(p, c["payload_strand"])
        rows.append({
            "method": method, "arm": arm, "deletion_rate": level,
            "cluster_id": c["cluster_id"], "pair_id": c["pair_id"],
            "file_id": "%s_file%03d" % (G.level_tag(level), i // FILE_SIZE),
            "payload_length": len(c["payload_strand"]),
            "true_payload": c["payload_strand"], "pred_payload": p,
            "markers_found": "" if markers_found is None else round(markers_found[i], 3), **s,
        })
    return rows


def run_mv(method, arm, clusters):
    t0 = time.time()
    if method == "mv":
        preds, found = [G.remove_markers(arm, mv_plain(c["noisy_reads"])) for c in clusters], None
    else:
        out = [mv_sync(c["noisy_reads"]) for c in clusters]
        preds, found = [p for p, _ in out], [f for _, f in out]
    return preds, found, time.time() - t0


def run_dnaformer(model, arm, clusters, device):
    import torch
    if device == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    preds = []
    with torch.inference_mode():
        for i in range(0, len(clusters), DNAF_BATCH):
            chunk = clusters[i:i + DNAF_BATCH]
            x, _ = A.make_batch(chunk, "relaxed", device)
            logits = model(x)["pred"]
            if not torch.isfinite(logits).all():
                raise RuntimeError("DNAFormer produced NaN/inf outputs")
            preds += [G.remove_markers(arm, p) for p in A.decode(logits)]
    if device == "cuda":
        torch.cuda.synchronize()
    runtime = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else ""
    return preds, runtime, peak


def write_csv(path, rows, fields):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in rows:
            w.writerow({k: (round(v, 6) if isinstance(v, float) else v) for k, v in r.items()})


def print_table(results):
    print("\n%-12s %-16s %5s  %-22s %-8s %-8s %-6s %s" % ("method", "arm", "del", "exact recovery [95% CI]",
                                                      "NED", "base acc", "files", "ms/cl"))
    for r in results:
        print("%-12s %-16s %4.0f%%  %5.1f%% [%5.1f, %5.1f]    %.4f   %.4f   %2d/%-3d %.1f" % (
            r["method"], r["arm"], 100 * r["deletion_rate"], 100 * r["exact_recovery"],
            100 * r["exact_ci_low"], 100 * r["exact_ci_high"], r["ned_mean"], r["base_acc_mean"],
            r["files_recovered"], r["n_files"], r["ms_per_cluster"]))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--ckpt", default=None)
    ap.add_argument("--out", required=True)
    ap.add_argument("--limit", type=int, default=0, help="clusters per level (0 = all)")
    ap.add_argument("--no-torch", action="store_true", help="majority votes only")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    levels = G.DELETION_LEVELS
    log = {"python": platform.python_version(), "file_size": FILE_SIZE, "marker_max_ed": MARKER_MAX_ED,
           "marker_search": MARKER_SEARCH, "read_policy": "relaxed", "limit": args.limit}

    model, device = None, "cpu"
    if not args.no_torch:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        log["torch"] = torch.__version__
        log["device"] = device
        if device == "cuda":
            log["gpu"] = torch.cuda.get_device_name(0)
        else:
            print("WARNING: no GPU, DNAFormer will be slow. Runtime > Change runtime type > T4 GPU.")
        if not args.ckpt or not os.path.isfile(args.ckpt):
            sys.exit("ERROR: checkpoint not found: %s" % args.ckpt)
        model, _ = A.build_model(args.repo, device)
        rep = A.load_checkpoint(model, args.ckpt, device)
        model.eval()
        print("Checkpoint loaded strictly: %d tensors, renamed %s" % (rep["n_tensors"], rep["renamed"]))
        log["checkpoint"] = {k: rep[k] for k in ("loader", "n_tensors", "checkpoint_epoch", "renamed")}

    runs = [("mv", arm) for arm in G.ARMS] + [("mv_sync", "marker_periodic")]
    if model is not None:
        runs += [("dnaformer", arm) for arm in G.ARMS]

    all_rows, results = [], []
    for method, arm in runs:
        for level in levels:
            clusters = load_test(args.data, arm, level, args.limit)
            found = None
            if method == "dnaformer":
                preds, runtime, peak = run_dnaformer(model, arm, clusters, device)
            else:
                preds, found, runtime = run_mv(method, arm, clusters)
                peak = ""
            rows = make_rows(method, arm, level, clusters, preds, found)
            res = summarize(method, arm, level, rows, runtime, peak)
            all_rows += rows
            results.append(res)
            print("  done %-10s %-16s %2.0f%%: exact %5.1f%%, NED %.4f (%.1fs)" % (
                method, arm, 100 * level, 100 * res["exact_recovery"], res["ned_mean"], runtime), flush=True)

    write_csv(os.path.join(args.out, "predictions.csv"), all_rows, PRED_FIELDS)
    write_csv(os.path.join(args.out, "results.csv"), results, list(results[0].keys()))
    with open(os.path.join(args.out, "stage3_log.json"), "w") as f:
        json.dump(log, f, indent=1)

    print_table(results)
    if model is not None and device == "cuda":
        peaks = [r["gpu_peak_gb"] for r in results if r["method"] == "dnaformer"]
        print("\nDNAFormer inference peak GPU memory: %.2f GB (batch %d)" % (max(peaks), DNAF_BATCH))
    mf = [r for r in results if r["method"] == "mv_sync"]
    if mf:
        print("Markers found by mv_sync (share of 4 markers x 10 reads): " +
              ", ".join("%.0f%%: %.1f%%" % (100 * r["deletion_rate"], 100 * r["markers_found"]) for r in mf))
    n_expected = len(runs) * len(levels)
    ok = len(results) == n_expected and all(r["n_clusters"] == results[0]["n_clusters"] for r in results)
    print("\nSaved %d prediction rows and %d result rows to %s" % (len(all_rows), len(results), args.out))
    print("STAGE 3 DONE" if ok else "STAGE 3 INCOMPLETE: expected %d result rows" % n_expected)


if __name__ == "__main__":
    main()

### Cell 7: write the Stage 5 analysis script

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/stage5_analysis.py
"""
Stage 5: final evaluation, statistics, marker ablation and explainability (XAI).

Parts (run separately; each writes into <out>/<part>/):
  stats     CPU. Reads every Stage 4 run's predictions.csv and results.csv and makes
            - results_all.csv           every run x deletion level (exact, NED, data recovered)
            - paired_tests.csv          McNemar exact test on the same test strands + paired bootstrap
                                        CI of the data-recovered difference, for the planned comparisons
            - fig_*.png / *.csv         recovery, NED and data recovered vs deletion rate per study
            - error_position_*.png/csv  where along the strand the errors happen (8% deletion)
  ablation  GPU. Rebuilds each test strand with its markers replaced by random bases, re-simulates
            the reads with the SAME noise seed (same error positions), runs the fine-tuned model and
            compares with the normal strands: does the model need the markers?
  xai       GPU. Attribution: which read positions the fine-tuned model relies on.
            - gradient x input for every output letter -> heatmap (output position x strand position)
            - Integrated Gradients and GradientSHAP (Captum) on fewer strands, output in 8 segments
            - "marker attribution share" vs the share of the strand the markers occupy
            - attention: share of transformer attention that goes to marker positions
            - example figure: one read with its letters shaded by importance
Usage:
  python stage5_analysis.py --part stats    --results R --out O
  python stage5_analysis.py --part ablation --repo REPO --project P --out O --arms a,b
  python stage5_analysis.py --part xai      --repo REPO --project P --out O --arms a,b
"""
import argparse
import csv
import json
import math
import os
import random
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import dnaf_adapter as A  # noqa: E402
import stage1_generate as G  # noqa: E402

LEVELS = G.DELETION_LEVELS
STRAND = G.STRAND_LEN
B_BOOT = 5000

# Planned comparisons (A, B): "is A different from B?"
COMPARISONS = [
    ("marker_periodic", "marker_endblock", "spacing: periodic vs end-block (same content)"),
    ("marker_periodic", "marker_free", "8-base markers vs no markers"),
    ("marker_endblock", "marker_free", "end-block vs no markers (payload effect)"),
    ("marker_len04", "marker_free", "4-base vs no markers"),
    ("marker_len05", "marker_free", "5-base vs no markers"),
    ("marker_len06", "marker_free", "6-base vs no markers"),
    ("marker_len07", "marker_free", "7-base vs no markers"),
    ("marker_len12", "marker_free", "12-base vs no markers"),
    ("marker_len06", "marker_periodic", "6-base vs 8-base"),
    ("marker_len12", "marker_periodic", "12-base vs 8-base"),
    ("marker_len06_n1", "marker_free", "1 marker vs no markers"),
    ("marker_len06_n2", "marker_free", "2 markers vs no markers"),
    ("marker_len06_n8", "marker_free", "8 markers vs no markers"),
    ("marker_len06_n1", "marker_len06", "1 marker vs 4 markers"),
    ("marker_len06_n8", "marker_len06", "8 markers vs 4 markers"),
    ("marker_len06_n1", "marker_len06_n8", "1 marker vs 8 markers"),
    ("marker_len06_n8", "marker_len12", "8 x 6 vs 4 x 12 (same space)"),
    ("marker_free_seed2", "marker_free", "repeat run: no markers seed 2 vs 1"),
    ("marker_len06_seed2", "marker_len06", "repeat run: 6-base seed 2 vs 1"),
    ("marker_periodic_seed2", "marker_periodic", "repeat run: 8-base seed 2 vs 1"),
]
STUDIES = {
    "main": ["marker_free", "marker_periodic", "marker_endblock"],
    "lengths": ["marker_free", "marker_len04", "marker_len05", "marker_len06", "marker_len07",
                "marker_periodic", "marker_len12"],
    "spacing": ["marker_free", "marker_len06_n1", "marker_len06_n2", "marker_len06", "marker_len06_n8"],
}
LABEL = {"marker_free": "no markers", "marker_periodic": "8-base x4", "marker_endblock": "end-block 8-base x4",
         "marker_len04": "4-base x4", "marker_len05": "5-base x4", "marker_len06": "6-base x4",
         "marker_len07": "7-base x4", "marker_len12": "12-base x4", "marker_len06_n1": "6-base x1 (middle)",
         "marker_len06_n2": "6-base x2", "marker_len06_n8": "6-base x8"}


def arm_of(run):
    return run.split("_seed")[0]


def data_dir(project, arm):
    sub = ("stage1_pilot" if arm in G.ARMS else "stage1_spacing" if arm in G.SPACING_ARMS else
           "stage1_middle" if arm in G.MIDDLE_ARMS else "stage1_markers")
    return os.path.join(project, "data", sub)


def marker_mask(arm):
    """Boolean list over the 128 strand positions: True where the arm has a marker base."""
    _, _, pos = G.build_strand(arm, "A" * STRAND)
    m = [False] * STRAND
    for a, b in pos:
        for i in range(a, b):
            m[i] = True
    return m


def payload_to_strand_positions(arm):
    mm = marker_mask(arm)
    return [i for i in range(STRAND) if not mm[i]]


def read_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))


def write_csv(path, rows, fields=None):
    if not rows:
        return
    fields = fields or list(rows[0].keys())
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in rows:
            w.writerow({k: (round(v, 6) if isinstance(v, float) else v) for k, v in r.items()})


def mcnemar_exact(b, c):
    """Two-sided exact McNemar p-value (binomial test on the discordant pairs)."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * tail)


def plt_setup():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                         "figure.dpi": 150, "savefig.bbox": "tight"})
    return plt


# ===========================================================================
# Part 1: statistics and final graphs
# ===========================================================================
def part_stats(args):
    out = os.path.join(args.out, "stats")
    os.makedirs(out, exist_ok=True)
    runs = sorted(d for d in os.listdir(args.results)
                  if os.path.isfile(os.path.join(args.results, d, "predictions.csv"))
                  and arm_of(d) in G.ARM_SPECS)
    print("Runs found (%d): %s" % (len(runs), ", ".join(runs)))
    preds = {}
    for run in runs:
        rows = read_csv(os.path.join(args.results, run, "predictions.csv"))
        by = {}
        for r in rows:
            by.setdefault(float(r["deletion_rate"]), {})[r["pair_id"]] = r
        preds[run] = by

    # ---- results_all.csv
    allrows = []
    for run in runs:
        for d in LEVELS:
            rr = list(preds[run].get(d, {}).values())
            if not rr:
                continue
            n = len(rr)
            k = sum(int(r["exact"]) for r in rr)
            plen = int(rr[0]["payload_length"])
            lo, hi = wilson(k, n)
            allrows.append({"run": run, "arm": arm_of(run), "deletion_rate": d, "n": n, "data_bases": plen,
                            "overhead": 1 - plen / STRAND, "exact": k / n, "exact_ci_low": lo, "exact_ci_high": hi,
                            "ned": float(np.mean([float(r["ned"]) for r in rr])),
                            "data_recovered_per_1000": k * plen * 1000 / n})
    write_csv(os.path.join(out, "results_all.csv"), allrows)

    # ---- paired tests
    tests = []
    rng = np.random.default_rng(0)
    for a, b, what in COMPARISONS:
        if a not in preds or b not in preds:
            continue
        for d in (0.05, 0.08):
            pa, pb = preds[a].get(d, {}), preds[b].get(d, {})
            ids = sorted(set(pa) & set(pb))
            if not ids:
                continue
            ea = np.array([int(pa[i]["exact"]) for i in ids])
            eb = np.array([int(pb[i]["exact"]) for i in ids])
            la, lb = int(pa[ids[0]]["payload_length"]), int(pb[ids[0]]["payload_length"])
            bb, cc = int(((ea == 1) & (eb == 0)).sum()), int(((ea == 0) & (eb == 1)).sum())
            diff = ea * la - eb * lb                      # data bases recovered, per strand
            boot = np.array([diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(B_BOOT)])
            tests.append({"A": a, "B": b, "comparison": what, "deletion_rate": d, "n_pairs": len(ids),
                          "exact_A": ea.mean(), "exact_B": eb.mean(), "only_A_correct": bb, "only_B_correct": cc,
                          "mcnemar_p": mcnemar_exact(bb, cc),
                          "data_diff_per_1000": diff.mean() * 1000,
                          "data_diff_ci_low": np.percentile(boot, 2.5) * 1000,
                          "data_diff_ci_high": np.percentile(boot, 97.5) * 1000,
                          "data_diff_pct_of_B": diff.mean() / max(1e-9, (eb * lb).mean()) * 100})
    write_csv(os.path.join(out, "paired_tests.csv"), tests)

    # ---- graphs per study
    plt = plt_setup()
    res = {(r["run"], r["deletion_rate"]): r for r in allrows}
    for study, arms in STUDIES.items():
        arms = [a for a in arms if a in preds]
        if len(arms) < 2:
            continue
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
        rows_csv = []
        for ai, a in enumerate(arms):
            dodge = (ai - (len(arms) - 1) / 2) * 0.07
            xs = [d * 100 + dodge for d in LEVELS if (a, d) in res]
            ex = [res[(a, d)]["exact"] * 100 for d in LEVELS if (a, d) in res]
            lo = [res[(a, d)]["exact_ci_low"] * 100 for d in LEVELS if (a, d) in res]
            hi = [res[(a, d)]["exact_ci_high"] * 100 for d in LEVELS if (a, d) in res]
            ned = [res[(a, d)]["ned"] for d in LEVELS if (a, d) in res]
            dr = [res[(a, d)]["data_recovered_per_1000"] / 1000 for d in LEVELS if (a, d) in res]
            kw = dict(marker="o", ms=3.5, lw=1.6 if a != "marker_free" else 2.2,
                      ls="--" if a == "marker_free" else "-", label=LABEL.get(a, a))
            axes[0].errorbar(xs, ex, yerr=[np.subtract(ex, lo), np.subtract(hi, ex)], capsize=2, **kw)
            axes[1].plot(xs, ned, **kw)
            axes[2].plot(xs, dr, **kw)
            for d, e, n_, r_ in zip(xs, ex, ned, dr):
                rows_csv.append({"study": study, "arm": a, "deletion_pct": round(d), "exact_pct": e, "ned": n_,
                                 "data_bases_per_strand": r_})
        axes[0].set(title="Strands fully recovered", xlabel="Deletion rate (%)", ylabel="Exact recovery (%)",
                    xticks=[1, 3, 5, 8])
        axes[1].set(title="Letter error (NED, lower is better)", xlabel="Deletion rate (%)", ylabel="NED",
                    xticks=[1, 3, 5, 8])
        axes[2].set(title="Data recovered per strand", xlabel="Deletion rate (%)",
                    ylabel="Data bases recovered / strand", xticks=[1, 3, 5, 8])
        h, l = axes[0].get_legend_handles_labels()
        fig.legend(h, l, loc="lower center", ncol=min(7, len(l)), frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.06))
        fig.suptitle("Fine-tuned DNAFormer, %s study (1,000 test strands per point, 95%% CI)" % study, fontsize=10, y=1.03)
        fig.tight_layout()
        fig.savefig(os.path.join(out, "fig_%s_vs_deletion.png" % study))
        plt.close(fig)
        write_csv(os.path.join(out, "fig_%s_vs_deletion.csv" % study), rows_csv)

    # ---- error position along the strand (8% deletion)
    for study, arms in STUDIES.items():
        arms = [a for a in arms if a in preds and 0.08 in preds[a]]
        if not arms:
            continue
        fig, ax = plt.subplots(figsize=(10, 3.2))
        rows_csv = []
        for a in arms:
            rr = list(preds[a][0.08].values())
            pos = payload_to_strand_positions(a)
            err = np.zeros(STRAND)
            cnt = np.zeros(STRAND)
            for r in rr:
                t, p = r["true_payload"], r["pred_payload"]
                for i, s in enumerate(pos):
                    cnt[s] += 1
                    err[s] += t[i] != p[i]
            rate = np.where(cnt > 0, err / np.maximum(cnt, 1), np.nan)
            ax.plot(range(1, STRAND + 1), rate * 100, lw=1.3 if a != "marker_free" else 2,
                    ls="--" if a == "marker_free" else "-", label=LABEL.get(a, a))
            for s in range(STRAND):
                rows_csv.append({"arm": a, "strand_position": s + 1,
                                 "error_pct": None if np.isnan(rate[s]) else rate[s] * 100})
        ref = {"spacing": "marker_len06_n8", "lengths": "marker_len06"}.get(study, "marker_periodic")
        mm = marker_mask(ref)
        for s in range(STRAND):
            if mm[s]:
                ax.axvspan(s + 0.5, s + 1.5, color="#dddddd", lw=0)
        ax.set(xlabel="Position in the 128-base strand (gaps = a version's own marker letters, not scored)",
               ylabel="Wrong letters (%)", xlim=(1, STRAND),
               title="Where the errors are at 8%% deletion, %s study (grey = marker positions of %s)"
                     % (study, LABEL.get(ref, ref)))
        ax.legend(fontsize=7, frameon=False, ncol=3)
        fig.savefig(os.path.join(out, "error_position_%s.png" % study))
        plt.close(fig)
        write_csv(os.path.join(out, "error_position_%s.csv" % study), rows_csv)

    # ---- print summary
    print("\nPAIRED TESTS (McNemar on the same test strands; data difference with 95% bootstrap CI)")
    print("%-42s %4s  %7s %7s  %10s  %22s" % ("comparison", "del", "A", "B", "McNemar p", "data diff / 1000 strands"))
    for t in tests:
        print("%-42s %3.0f%%  %6.1f%% %6.1f%%  %10.2g  %+8.0f [%+.0f, %+.0f]" % (
            t["comparison"][:42], t["deletion_rate"] * 100, t["exact_A"] * 100, t["exact_B"] * 100,
            t["mcnemar_p"], t["data_diff_per_1000"], t["data_diff_ci_low"], t["data_diff_ci_high"]))
    print("\nSaved to %s" % out)
    print("STAGE 5 STATS DONE")


def wilson(k, n, z=1.959964):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    den = 1 + z * z / n
    mid = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return max(0.0, mid - half), min(1.0, mid + half)


# ===========================================================================
# Shared GPU helpers
# ===========================================================================
def load_finetuned(args, run, device):
    import torch
    model, _ = A.build_model(args.repo, device)
    path = os.path.join(args.project, "checkpoints", "stage4", run, "best.pt")
    if not os.path.isfile(path):
        sys.exit("ERROR: fine-tuned model not found: %s" % path)
    st = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(st["model"], strict=True)
    model.eval()
    return model, st.get("epoch")


def load_test(project, arm, level, limit):
    with open(os.path.join(data_dir(project, arm), arm, "test_" + G.level_tag(level), "metadata.json")) as f:
        cl = json.load(f)
    return cl[:limit] if limit else cl


def predict(torch, model, arm, clusters, device, bs=32):
    out = []
    with torch.inference_mode():
        for i in range(0, len(clusters), bs):
            x, _ = A.make_batch(clusters[i:i + bs], "relaxed", device)
            out += [G.remove_markers(arm, p) for p in A.decode(model(x)["pred"])]
    return out


# ===========================================================================
# Part 2: marker ablation
# ===========================================================================
def scrambled_cluster(CG, arm, c, seed):
    """Same payload and noise seed, but every marker base replaced by a different random base."""
    rng = random.Random(seed)
    clean = list(c["clean_strand"])
    for a, b in c["marker_positions"]:
        for i in range(a, b):
            clean[i] = rng.choice([x for x in "ACGT" if x != clean[i]])
    clean = "".join(clean)
    reads = G.simulate_reads(CG, clean, c["deletion_rate"], c["noise_seed"])
    return dict(c, clean_strand=clean, noisy_reads=reads)


def part_ablation(args):
    import torch
    out = os.path.join(args.out, "ablation")
    os.makedirs(out, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    CG = G.load_simulator(args.repo)
    rows = []
    for run in args.arms:
        arm = arm_of(run)
        if G.ARM_SPECS[arm]["layout"] == "none":
            continue
        model, ep = load_finetuned(args, run, device)
        for d in (0.05, 0.08):
            cl = load_test(args.project, arm, d, args.limit)
            # sanity: re-simulating the unchanged strand must give the stored reads exactly
            chk = [G.simulate_reads(CG, c["clean_strand"], c["deletion_rate"], c["noise_seed"]) == c["noisy_reads"]
                   for c in cl[:20]]
            if not all(chk):
                sys.exit("ERROR: re-simulation does not reproduce the stored reads; ablation would be unfair")
            t0 = time.time()
            scr = [scrambled_cluster(CG, arm, c, 7 + k) for k, c in enumerate(cl)]
            p_norm = predict(torch, model, arm, cl, device)
            p_scr = predict(torch, model, arm, scr, device)
            en = np.array([p == c["payload_strand"] for p, c in zip(p_norm, cl)], dtype=int)
            es = np.array([p == c["payload_strand"] for p, c in zip(p_scr, cl)], dtype=int)
            nn_ = np.mean([A.edit_distance(p, c["payload_strand"]) / len(c["payload_strand"]) for p, c in zip(p_norm, cl)])
            ns = np.mean([A.edit_distance(p, c["payload_strand"]) / len(c["payload_strand"]) for p, c in zip(p_scr, cl)])
            bb, cc = int(((en == 1) & (es == 0)).sum()), int(((en == 0) & (es == 1)).sum())
            r = {"run": run, "deletion_rate": d, "n": len(cl), "exact_normal": en.mean(), "exact_scrambled": es.mean(),
                 "drop_points": (en.mean() - es.mean()) * 100, "ned_normal": nn_, "ned_scrambled": ns,
                 "mcnemar_p": mcnemar_exact(bb, cc), "best_epoch": ep, "seconds": time.time() - t0}
            rows.append(r)
            print("  %-22s %2.0f%%: exact %5.1f%% -> %5.1f%% with markers scrambled (drop %+.1f pts, p=%.2g), "
                  "NED %.4f -> %.4f" % (run, d * 100, r["exact_normal"] * 100, r["exact_scrambled"] * 100,
                                        -r["drop_points"], r["mcnemar_p"], nn_, ns), flush=True)
        del model
        if device == "cuda":
            torch.cuda.empty_cache()
    write_csv(os.path.join(out, "ablation.csv"), rows)
    if rows:
        plt = plt_setup()
        r8 = [r for r in rows if r["deletion_rate"] == 0.08]
        fig, ax = plt.subplots(figsize=(max(4, 1.3 * len(r8)), 3.2))
        x = np.arange(len(r8))
        ax.bar(x - 0.18, [r["exact_normal"] * 100 for r in r8], 0.36, label="markers intact", color="#2a6fd6")
        ax.bar(x + 0.18, [r["exact_scrambled"] * 100 for r in r8], 0.36, label="markers scrambled", color="#bbbbbb")
        ax.set_xticks(x, [LABEL.get(arm_of(r["run"]), r["run"]) for r in r8], rotation=20, ha="right")
        ax.set(ylabel="Strands fully recovered (%)", title="Marker ablation at 8% deletion")
        ax.legend(frameon=False, fontsize=8)
        fig.savefig(os.path.join(out, "ablation_8pct.png"))
        plt.close(fig)
    print("\nSaved to %s\nSTAGE 5 ABLATION DONE" % out)


# ===========================================================================
# Part 3: XAI
# ===========================================================================
def align_read_to_strand(strand, read):
    """Edit-distance alignment; returns, for each read position, the strand position it aligns to (or -1)."""
    n, m = len(strand), len(read)
    D = np.zeros((n + 1, m + 1), dtype=np.int32)
    D[:, 0] = np.arange(n + 1)
    D[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        si = strand[i - 1]
        for j in range(1, m + 1):
            D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + (si != read[j - 1]))
    mp = [-1] * m
    i, j = n, m
    while i > 0 and j > 0:
        if D[i, j] == D[i - 1, j - 1] + (strand[i - 1] != read[j - 1]):
            mp[j - 1] = i - 1
            i, j = i - 1, j - 1
        elif D[i, j] == D[i - 1, j] + 1:
            i -= 1
        else:
            j -= 1
    return mp


def read_importance(att, reads):
    """att: [2, 16, 4, 132] attribution for one cluster (left branch, reversed right branch).
    Returns per read a list of |importance| per read position (both branches combined)."""
    a = np.abs(att.sum(axis=2))            # [2, 16, 132]
    out = []
    for i, r in enumerate(reads[:A.MAX_READS]):
        L = min(len(r), A.INPUT_LEN)
        left = a[0, i, :L]
        right = a[1, i, :L][::-1]          # right branch holds the reversed read
        out.append(left + right)
    return out


def strand_profile(imp_reads, maps):
    prof = np.zeros(STRAND)
    for imp, mp in zip(imp_reads, maps):
        for j, v in enumerate(imp):
            if j < len(mp) and mp[j] >= 0:
                prof[mp[j]] += v
    return prof


def part_xai(args):
    import torch
    out = os.path.join(args.out, "xai")
    os.makedirs(out, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    plt = plt_setup()
    summary = []
    heatmaps = {}
    for run in args.arms:
        arm = arm_of(run)
        model, ep = load_finetuned(args, run, device)
        for p in model.parameters():
            p.requires_grad_(False)
        cl = load_test(args.project, arm, 0.08, args.n_xai)
        mm = np.array(marker_mask(arm))
        ref_mm = mm if mm.any() else np.array(marker_mask("marker_len06"))   # placebo positions for no markers
        pay_out = [s for s in range(STRAND) if not mm[s]]                    # output positions holding data
        t0 = time.time()
        H = np.zeros((STRAND, STRAND))
        shares, local = [], []
        K = args.k_batch
        band = np.array([[abs(o - s) <= 8 for s in range(STRAND)] for o in pay_out])
        for c in cl:
            x, y = A.make_batch([c], "relaxed", device)                      # x [2, 16, 4, 132]
            maps = [align_read_to_strand(c["clean_strand"], r[:A.INPUT_LEN]) for r in c["noisy_reads"]]
            Hc = np.zeros((STRAND, STRAND))
            for s in range(0, STRAND, K):
                tg = list(range(s, min(STRAND, s + K)))
                k = len(tg)
                X = torch.cat([x[0:1].repeat(k, 1, 1, 1), x[1:2].repeat(k, 1, 1, 1)]).clone().requires_grad_(True)
                with torch.enable_grad():
                    pred = model(X)["pred"]                                   # [k, 4, 128]
                    score = sum(pred[i, int(y[0, j]), j] for i, j in enumerate(tg))
                    g, = torch.autograd.grad(score, X)
                gx = (g * X).detach().float().cpu().numpy()                   # gradient x input
                for i, j in enumerate(tg):
                    att = np.stack([gx[i], gx[k + i]])                       # [2, 16, 4, 132]
                    prof = strand_profile(read_importance(att, c["noisy_reads"]), maps)
                    Hc[j] = prof / max(prof.sum(), 1e-12)
            H += Hc
            rows_pay = Hc[pay_out]
            shares.append(rows_pay[:, ref_mm].sum(axis=1).mean())
            local.append((rows_pay * band).sum(axis=1).mean())
        H /= len(cl)
        heatmaps[run] = (H, mm)
        np.save(os.path.join(out, "heatmap_%s.npy" % run), H)

        # ---- Integrated Gradients and GradientSHAP on fewer strands, 8 output segments
        ig_sh, gs_sh = [], []
        try:
            from captum.attr import GradientShap, IntegratedGradients
            segs = [pay_out[k * len(pay_out) // 8:(k + 1) * len(pay_out) // 8] for k in range(8)]

            def fwd(z, yy):
                p = model(torch.cat([z[:, 0], z[:, 1]]))["pred"]               # [N, 4, 128]
                sc = p.gather(1, yy.unsqueeze(1).expand(p.shape[0], 1, STRAND)).squeeze(1)  # [N, 128]
                return torch.stack([sc[:, sg].sum(1) for sg in segs], 1)      # [N, 8]
            for c in cl[:args.n_ig]:
                x, y = A.make_batch([c], "relaxed", device)
                z = torch.stack([x[0], x[1]]).unsqueeze(0)                    # [1, 2, 16, 4, 132]
                maps = [align_read_to_strand(c["clean_strand"], r[:A.INPUT_LEN]) for r in c["noisy_reads"]]
                ig = IntegratedGradients(lambda zz: fwd(zz, y.expand(zz.shape[0], -1)))
                gs = GradientShap(lambda zz: fwd(zz, y.expand(zz.shape[0], -1)))
                for t in range(8):
                    with torch.enable_grad():
                        a1 = ig.attribute(z, baselines=torch.zeros_like(z), target=t, n_steps=args.ig_steps,
                                          internal_batch_size=8)
                        a2 = gs.attribute(z, baselines=torch.zeros_like(z), target=t,
                                          n_samples=min(args.ig_steps, 16), stdevs=0.0)
                    for a_, lst in ((a1, ig_sh), (a2, gs_sh)):
                        prof = strand_profile(read_importance(a_[0].detach().float().cpu().numpy(), c["noisy_reads"]), maps)
                        lst.append(prof[ref_mm].sum() / max(prof.sum(), 1e-12))
        except ImportError:
            print("  (captum not installed: skipping Integrated Gradients / GradientSHAP)")

        # ---- attention share to marker positions
        att_share = attention_share(torch, model, cl[:args.n_att], ref_mm, pay_out, device)
        r = {"run": run, "best_epoch": ep, "n_strands": len(cl),
             "marker_space_share": ref_mm.mean(), "placebo_positions": not mm.any(),
             "gradxinput_marker_share": float(np.mean(shares)), "gradxinput_local_share_pm8": float(np.mean(local)),
             "ig_marker_share": float(np.mean(ig_sh)) if ig_sh else None,
             "gradshap_marker_share": float(np.mean(gs_sh)) if gs_sh else None,
             "attention_marker_share": att_share, "seconds": time.time() - t0}
        r["enrichment_gradxinput"] = r["gradxinput_marker_share"] / r["marker_space_share"]
        summary.append(r)
        print("  %-22s marker positions = %4.1f%% of strand%s | importance on them: grad x input %4.1f%% "
              "(x%.2f), IG %s, GradSHAP %s, attention %s | %.0fs" % (
                  run, r["marker_space_share"] * 100, " (placebo: no markers)" if not mm.any() else "",
                  r["gradxinput_marker_share"] * 100, r["enrichment_gradxinput"],
                  "%.1f%%" % (r["ig_marker_share"] * 100) if ig_sh else "-",
                  "%.1f%%" % (r["gradshap_marker_share"] * 100) if gs_sh else "-",
                  "%.1f%%" % (att_share * 100) if att_share is not None else "-", r["seconds"]), flush=True)
        if run == args.arms[0] or mm.any():
            example_figure(plt, torch, model, arm, run, cl[0], device, out)
        del model
        if device == "cuda":
            torch.cuda.empty_cache()
    write_csv(os.path.join(out, "xai_summary.csv"), summary)
    # heatmaps
    n = len(heatmaps)
    if n:
        fig, axes = plt.subplots(1, n, figsize=(3.3 * n, 3.3), squeeze=False)
        for ax, (run, (H, mm)) in zip(axes[0], heatmaps.items()):
            ax.imshow(H, cmap="magma", origin="upper", vmax=np.percentile(H, 99.5), aspect="equal")
            for s in np.where(mm)[0]:
                ax.plot([s, s], [-6, -1.5], color="#e8590c", lw=1.2, clip_on=False)
            ax.set_ylim(STRAND - 0.5, -7)
            ax.set(title=LABEL.get(arm_of(run), run), xlabel="Strand position read by the model", ylabel="Output letter")
        fig.suptitle("Where the fine-tuned model looks (gradient x input, 8% deletion; orange ticks = marker positions)",
                     fontsize=9)
        fig.savefig(os.path.join(out, "heatmaps.png"))
        plt.close(fig)
        fig, ax = plt.subplots(figsize=(max(4, 1.2 * len(summary)), 3.2))
        x = np.arange(len(summary))
        ax.bar(x - 0.2, [r["marker_space_share"] * 100 for r in summary], 0.4, color="#cccccc", label="share of strand")
        ax.bar(x + 0.2, [r["gradxinput_marker_share"] * 100 for r in summary], 0.4, color="#2a6fd6",
               label="share of model's attention (grad x input)")
        ax.set_xticks(x, [LABEL.get(arm_of(r["run"]), r["run"]) for r in summary], rotation=20, ha="right")
        ax.set(ylabel="%", title="Marker positions: space they take vs importance the model gives them")
        ax.legend(frameon=False, fontsize=7)
        fig.savefig(os.path.join(out, "marker_share.png"))
        plt.close(fig)
    print("\nSaved to %s\nSTAGE 5 XAI DONE" % out)


def attention_share(torch, model, clusters, ref_mm, pay_out, device):
    """Average share of self-attention (all layers, heads averaged) that data positions give to marker positions."""
    store = []
    origs = []
    for layer in model.encoder.layers:
        mha = layer.self_attn
        orig = mha.forward

        def patched(*a, _orig=orig, **kw):
            kw["need_weights"] = True
            kw["average_attn_weights"] = True
            o = _orig(*a, **kw)
            store.append(o[1].detach())
            return o
        mha.forward = patched
        origs.append((mha, orig))
    shares = []
    try:
        for c in clusters:
            x, _ = A.make_batch([c], "relaxed", device)
            x = x.clone().requires_grad_(True)       # keeps the transformer off its fused fast path
            store.clear()
            with torch.enable_grad():
                model(x)
            if not store:
                return None
            for w in store:                          # [2, 128, 128] (left and right branch)
                w = w.float().cpu().numpy().mean(0)
                shares.append(w[pay_out][:, ref_mm].sum(1).mean())
    finally:
        for mha, orig in origs:
            mha.forward = orig
    return float(np.mean(shares)) if shares else None


def example_figure(plt, torch, model, arm, run, c, device, out):
    """One read with its letters shaded by importance (gradient x input summed over all output letters)."""
    x, y = A.make_batch([c], "relaxed", device)
    X = x.clone().requires_grad_(True)
    with torch.enable_grad():
        p = model(X)["pred"]
        score = p[0].gather(0, y[0].unsqueeze(0)).sum()
        g, = torch.autograd.grad(score, X)
    att = (g * X).detach().float().cpu().numpy()
    imp = read_importance(att, c["noisy_reads"])[0]
    read = c["noisy_reads"][0][:len(imp)]
    mp = align_read_to_strand(c["clean_strand"], read)
    mm = marker_mask(arm)
    v = imp / max(imp.max(), 1e-12)
    per_line = 66
    lines = math.ceil(len(read) / per_line)
    fig, ax = plt.subplots(figsize=(10, 0.55 * lines + 0.8))
    ax.axis("off")
    for j, ch in enumerate(read):
        r_, c_ = divmod(j, per_line)
        is_m = mp[j] >= 0 and mm[mp[j]]
        ax.add_patch(plt.Rectangle((c_, -r_ - 0.9), 0.95, 0.85, color=plt.cm.Blues(0.15 + 0.85 * v[j])))
        ax.text(c_ + 0.47, -r_ - 0.47, ch, ha="center", va="center", fontsize=7, family="monospace",
                color="white" if v[j] > 0.6 else "black", fontweight="bold" if is_m else "normal")
        if is_m:
            ax.plot([c_, c_ + 0.95], [-r_ - 0.95, -r_ - 0.95], color="#e8590c", lw=1.5)
    ax.set_xlim(0, per_line)
    ax.set_ylim(-lines - 0.1, 0.1)
    ax.set_title("%s: one noisy read, darker = the model relied on it more (orange underline = marker letters)"
                 % LABEL.get(arm, arm), fontsize=8)
    fig.savefig(os.path.join(out, "example_read_%s.png" % run))
    plt.close(fig)


# ===========================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--part", required=True, choices=["stats", "ablation", "xai"])
    ap.add_argument("--results", help="results/stage4 folder (stats)")
    ap.add_argument("--project", help="MyDrive/fyp_dna (ablation, xai)")
    ap.add_argument("--repo", help="Deep-DNA-based-storage (ablation, xai)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--arms", default="marker_free,marker_endblock,marker_len06,marker_len06_n1,marker_len06_n8",
                    help="comma-separated runs (folder names under checkpoints/stage4)")
    ap.add_argument("--limit", type=int, default=0, help="test strands per level for ablation (0 = all 1,000)")
    ap.add_argument("--n-xai", type=int, default=100, help="test strands (8%% deletion) for gradient x input")
    ap.add_argument("--n-ig", type=int, default=20, help="test strands for Integrated Gradients / GradientSHAP")
    ap.add_argument("--n-att", type=int, default=50, help="test strands for attention")
    ap.add_argument("--ig-steps", type=int, default=32)
    ap.add_argument("--k-batch", type=int, default=16, help="output letters explained per backward pass")
    args = ap.parse_args()
    args.arms = [a.strip() for a in args.arms.split(",") if a.strip()]
    bad = [a for a in args.arms if arm_of(a) not in G.ARM_SPECS]
    if bad:
        sys.exit("ERROR: unknown run(s) %s" % bad)
    os.makedirs(args.out, exist_ok=True)
    {"stats": part_stats, "ablation": part_ablation, "xai": part_xai}[args.part](args)


if __name__ == "__main__":
    main()

### Cell 8: get the DNAFormer repo (same pinned commit as before)

In [ ]:
!rm -rf {REPO}
!git clone -q https://github.com/itaiorr/Deep-DNA-based-storage {REPO}
!git -C {REPO} checkout -q 167a68261719ea458e79c40fcc485ae768c1a64a
!git -C {REPO} rev-parse HEAD

### Cell 9: Part A, statistics (a few minutes)
**Paste back** the tables printed above `STAGE 5 STATS DONE`.
Saves to `results/stage5/stats/`: `results_all.csv`, `paired_tests.csv`, `fig_*_vs_deletion.png`, `error_position_*.png`.

In [ ]:
if 'stats' in PARTS:
    !cd {CODE} && python stage5_analysis.py --part stats --results {PROJECT}/results/stage4 --out {OUT} 2>&1 | grep -v Warning
else:
    print('skipped (stats not in PARTS)')

### Cell 10: Part B, marker ablation (~10 min per model)
Same reads, same noise, but every marker letter is replaced by a different random letter.
If recovery drops, the model **uses** the markers. **Paste back** the table above `STAGE 5 ABLATION DONE`.

In [ ]:
if 'ablation' in PARTS:
    !cd {CODE} && python stage5_analysis.py --part ablation --project {PROJECT} --repo {REPO} --arms {ARMS} --out {OUT_BC} 2>&1 | grep -v Warning
else:
    print('skipped (ablation not in PARTS)')

### Cell 11: Part C, XAI (~30-45 min per model)
Gradient x input on 100 test strands (8% deletion), Integrated Gradients + GradientSHAP on 20, attention on 50.
**Paste back** the table above `STAGE 5 XAI DONE`.
Figures (`heatmaps.png`, `marker_share.png`, `example_read_*.png`) are saved in `results/stage5/<your ARMS>/xai/`.

In [ ]:
if 'xai' in PARTS:
    !cd {CODE} && python stage5_analysis.py --part xai --project {PROJECT} --repo {REPO} --arms {ARMS} --out {OUT_BC} 2>&1 | grep -v Warning
else:
    print('skipped (xai not in PARTS)')

### Cell 12: show the figures

In [ ]:
import glob
from IPython.display import Image, display
pngs = glob.glob(f'{OUT}/stats/*.png') if PARTS == ['stats'] else glob.glob(f'{OUT_BC}/**/*.png', recursive=True)
for p in sorted(pngs):
    print(p.replace(PROJECT + '/', '')); display(Image(p, width=900))

---
**How to read the XAI numbers.** *Marker share* = how much of the model's attention (importance) lands on marker
letters. *Space share* = how much of the strand the markers take up. **Share above space share** means the model
looks at the markers more than their size alone explains, i.e. it really uses them as anchors.